## Background

#### Description
Zachary (1977) reported observations of social relations in a university karate club, with membership varying between 50 and 100 individuals. Of these, 34 members - including 32 ordinary club members and officers, the club president ("John A."), and the part-time instructor ("Mr. Hi") - consistently interacted outside of the club.

During the study, the club divided into two factions and eventually split into *two separate clubs*: one led by *Mr. Hi* and the other by *John A.*. The split arose from a disagreement over whether *Mr. Hi* could unilaterally change the level of compensation for his services.

---

#### Format
The data are represented as a network object, with:

- An edge attribute `contexts`, giving the number of contexts of interaction between each pair of actors.
- The following vertex attributes:

| Attribute | Description |
|------------|-------------|
| `club` | The club in which the actor ultimately ended up. |
| `faction` | Faction alignment of the actor as recorded by Zachary. |
| `faction.id` | Faction alignment coded numerically as:<br>1 = strongly Mr. Hi's<br>2 = weakly Mr. Hi's<br>3 = neutral<br>4 = weakly John's<br>5 = strongly John's |
| `role` | Role of the actor in the network (`Instructor`, `Member`, or `President`). |

---

### Details
Zachary identified the factional alignment of each of the 34 actors and recorded the *count of social contexts* in which each pair of individuals interacted.  
The eight contexts of interaction were:

1. Academic classes at the university  
2. Hi's private karate studio (night classes)  
3. Hi's private karate studio (weekend classes)  
4. Student teaching at Hi's studio  
5. The university rathskeller (bar) near the karate club  
6. A bar located near the university campus  
7. Open karate tournaments in the area  
8. Intercollegiate karate tournaments  

The *maximum number of shared contexts* observed between any pair of individuals was 7.

---

#### Reference
Zachary, W. W. (1977). *An Information Flow Model for Conflict and Fission in Small Groups.*, Journal of Anthropological Research, 33(4), 452–473.

## Zachary's Karate Club - ERGRM Demo

This notebook demonstrates how to fit and analyze **count-valued Exponential-Family Random Graph Models (ERGMs)** using the R package **ergm.count**.  
The goal is to model and understand **network relationships where ties are not just present or absent**, but have **count values**. For example, the number of interactions or shared contexts between actors.

We use **Zachary’s Karate Club** network as an example. The analysis explores how node attributes (such as role and faction) and network structures influence the strength of ties, modeled as count data.

#### Step 1: Check whether the **ergm.count** package is installed in R.

The **ergm.count** package provides tools for modeling and analyzing count-valued networks using Exponential-Family Random Graph Models (ERGMs).

In [ ]:
pkgs <- c("ergm", "ergm.count", "network")  # 'ergm' for modeling, 'ergm.count' for count-valued ties
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) install.packages(p)
  library(p, character.only = TRUE)
}

#### Step 2: Load the dataset

In [ ]:
data("zach", package = "ergm.count")  # network with vertex/edge attributes
# Vertex attribute 'role' = node role; edge attribute 'contexts' = count (0..8)
# Vertex attribute 'faction.id' = faction membership

#### Step 3: Visualization of the network

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10)

if (!requireNamespace("RColorBrewer", quietly = TRUE)) install.packages("RColorBrewer")
library(RColorBrewer)

# --- Colors ---------------------------------------------------------------
roles <- get.vertex.attribute(zach, "role")
role_levels <- sort(unique(roles))
vcols <- setNames(brewer.pal(max(3, length(role_levels)), "Set2")[seq_along(role_levels)], role_levels)
vertex_colors <- vcols[as.character(roles)]

contexts <- get.edge.attribute(zach, "contexts")
n_breaks <- 7
edge_palette <- colorRampPalette(brewer.pal(9, "Blues"))(n_breaks)
edge_bins <- cut(contexts, breaks = n_breaks, include.lowest = TRUE)
edge_colors <- edge_palette[as.integer(edge_bins)]

# --- Plot layout: left = network, right = legends -------------------------
op <- par(no.readonly = TRUE)
on.exit({ par(op); layout(1) }, add = TRUE)

layout(matrix(c(1, 2), nrow = 1), widths = c(4, 2))  # widen left panel, narrow legend strip

## Left panel: network
par(mar = c(4, 4, 2.5, 1))  # standard margins
plot(
  zach,
  vertex.col    = vertex_colors,
  edge.col      = edge_colors,
  displaylabels = TRUE,
  main          = "Zachary Karate Club"
)

## Right panel: legends (own blank plotting region)
par(mar = c(4, 1, 2.5, 1))
plot.new()
plot.window(xlim = c(0, 1), ylim = c(0, 1))

legend(
  x = 0, y = 0.95,                    # top-left inside the legend strip
  legend = names(vcols),
  col = vcols,
  pch = 19,
  bty = "n",
  title = "Role",
  xjust = 0, yjust = 1
)

legend(
  x = 0, y = 0.55,                    # below the first legend
  legend = levels(edge_bins),
  col = edge_palette,
  lwd = 3,
  bty = "n",
  title = "Contexts",
  xjust = 0, yjust = 1
)

#### Step 4: Fitting a Valued ERGM with a binomial reference

Fit a **count-valued Exponential-Family Random Graph Model (ERGM)** to the Zachary's Karate Club network.

The model treats the edge attribute **contexts** as a **count variable** (ranging from 0 to 8) and uses a **Binomial(8)** reference distribution, meaning that each tie is modeled as a count bounded between 0 and 8.

The included terms capture different structural and attribute-based effects:
- **nonzero** - tendency for edges to be non-zero (presence of any tie)
- **sum** - overall intensity or total weight of edges in the network
- **nodefactor("role")** - influence of node role on tie strength
- **absdiffcat("faction.id")** - effect of faction differences on tie counts

After fitting, **MCMC diagnostics** are used to check model convergence, and the **summary** output provides parameter estimates and statistical significance for each effect.

In [ ]:
set.seed(123)  # reproducible MCMC

zach.fit1 <- ergm(
  zach ~
    nonzero +                           # tendency for any tie to be non-zero
    sum +                               # controls total edge weight across the network
    nodefactor("role", base = 2) +      # effect of node role (category 2 is reference)
    absdiffcat("faction.id"),           # effect of being in different factions
  response   = "contexts",              # model the count-valued edge attribute 'contexts'
  reference  = ~ Binomial(8)            # counts treated as Binomial with max 8
)

mcmc.diagnostics(zach.fit1)             # basic MCMC convergence checks
summary(zach.fit1)                      # coefficient estimates and tests

#### Step 5: Alternative Model Including Transitivity

In this step, an alternative **valued ERGM** is fitted that adds a **transitivity** term to the model.  
The **transitiveties** term captures the tendency for nodes to form **triadic or closed structures**. That is, if A is connected to B and B to C, there is a higher likelihood that A is also connected to C.  
This allows the model to account for **clustering** or **network closure** effects in tie formation.

The model skips the exact log-likelihood computation to improve performance, since this estimation is computationally intensive.

In [ ]:
# Note: This model is slower due to the transitivity term.
set.seed(123)
zach.fit2 <- ergm(
  zach ~ nonzero + sum + nodefactor("role", base = 2) + transitiveties,
  response    = "contexts",
  reference   = ~ Binomial(8),
  eval.loglik = FALSE                  # skip exact log-likelihood to save time
)

mcmc.diagnostics(zach.fit2)
summary(zach.fit2)